# AG_PRAXIS NB01 — Dataset Inventory

Everything else in this project rests on what is actually inside these files, so this
notebook reads them before anything gets built. The dataset paper describes the features
and the attacks, but a description is not the data, and I would rather meet a surprise
now than halfway through a training run.

The questions worth asking first are the ones that close options off. Is there a clock in
the data. Is there anything naming a sender or a receiver. Do all seventy-two files carry
the same columns. How are the recordings organised, and how many are there per attack.
How many rows does each attack have, and how far apart are the largest and the smallest.
And is there any column that sits still inside a recording while moving between
recordings, which would let a model tell the files apart rather than the attacks.

The whole pass is written as one function and run twice. The first pass is the fast one.
It does the cheap work only: finds the files, reads one of them, checks the column names
across all seventy-two, and parses every file name into a class and a recording. If a path
is wrong or my parsing rule breaks, it breaks there, in under a minute, instead of forty
minutes into reading eight million rows. The second pass does everything, including the
two steps that need every row.

The fast pass is a check on the code, not a measurement of the data, and nothing it
produces is ever entered in the ledger. The last cell puts the two passes side by side and
prints PASS or MISMATCH for every value both of them produced. Those values are structural
and cannot legitimately differ, so a mismatch there is a bug in the fast path rather than
something learned about the dataset.

Nothing is trained here. The full pass writes `dataset_inventory.json`, recording every
answer so no later notebook has to re-derive them or, worse, assume them, and
`config/feature_families.yaml`, a first grouping of the columns written as a draft because
it comes from spelling and nothing else.

The data sits on Drive and the code sits in the repository, so the first block mounts one
and clones the other. It also records the commit it is running from. Every number printed
below belongs to that commit, and if the working tree is dirty I would rather see it now
than when I am trying to explain a result later.

In [ ]:
import os
import subprocess
import sys
from datetime import date
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AGREWAL14/AG_PRAXIS.git"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/repo")
    if REPO_ROOT.exists():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "config" / "base.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args], capture_output=True, text=True
    ).stdout.strip()


GIT_SHA = git("rev-parse", "--short", "HEAD")
GIT_BRANCH = git("rev-parse", "--abbrev-ref", "HEAD")
GIT_DIRTY = bool(git("status", "--porcelain"))
RUN_DATE = date.today().isoformat()

print(f"colab     : {IN_COLAB}")
print(f"repo root : {REPO_ROOT}")
print(f"git sha   : {GIT_SHA} on {GIT_BRANCH}" + ("   WORKING TREE DIRTY" if GIT_DIRTY else ""))
print(f"run date  : {RUN_DATE}")

Paths and the seed come from `config/base.yaml` rather than from anything typed into a
cell, so changing a path is a commit and not an edit I forget I made.

The two passes write to two different folders, `NB01_fast` and `NB01`, so neither can
overwrite the other and I can always tell which set of files I am looking at. `NB01` is
the full pass and is the one the repository takes.

There is no `FAST` environment variable to set here. It used to select a one percent
sample of every file, and both passes now run from a single Run All, so the mode is an
argument to a function rather than something read from the environment. The sampling is
gone with it. Where a step cannot be done cheaply it is skipped outright rather than
approximated, because a column can hold one value through the first one percent of a file
and move through the rest, and an approximate answer to that question is worse than no
answer at all.

In [ ]:
import json
import random
import time

import numpy as np
import pandas as pd

from src import captures as cap
from src import inventory as inv

CFG = inv.load_config(REPO_ROOT)

SEED = CFG["seed"]
TRAIN_DIR = Path(CFG["paths"]["train_dir"])
TEST_DIR = Path(CFG["paths"]["test_dir"])
ARTIFACTS = Path(CFG["paths"]["artifacts"])
OUT_DIRS = {"fast": ARTIFACTS / "NB01_fast", "full": ARTIFACTS / "NB01"}

EXPECTED_FILES = 72

if IN_COLAB and not ARTIFACTS.exists():
    raise FileNotFoundError(
        f"{ARTIFACTS} does not exist. Drive is not mounted, or the artefacts path in "
        "config/base.yaml is wrong. Nothing this notebook writes would survive."
    )

pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

print(f"seed       : {SEED}")
print(f"train dir  : {TRAIN_DIR}   exists={TRAIN_DIR.exists()}")
print(f"test dir   : {TEST_DIR}   exists={TEST_DIR.exists()}")
print(f"fast pass  : {OUT_DIRS['fast']}")
print(f"full pass  : {OUT_DIRS['full']}   <- the one the repository takes")

No model is built here, but the seed is set before anything else runs so that any
sampling I do while looking at the data gives me the same rows the next time.

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
print(f"seeded with {SEED}")

What follows is one function per step, then a function that calls them in order, then the
cell that runs the whole thing twice. Splitting it this way keeps each step small enough
to read and lets the driver decide which ones a fast pass is allowed to take. Every step
prints its own results and returns them, so nothing has to be recomputed later and the
comparison at the end has something to compare.

Nothing below reads a data file until the driver cell runs. Defining a function does not
execute it, so the cells between here and there are cheap no matter what they contain.

In [ ]:
def section(title):
    print()
    print("-" * 79)
    print(title)
    print("-" * 79)


def skipped(title, reason="requires full data"):
    section(title)
    print(f"skipped, {reason}")

The first step finds the files and reads one of them all the way through, looking at every
column: the type it came back as, how many values are missing, how many distinct values it
takes, and where the numbers run from and to.

The distinct count is the part I watch hardest. A column holding one value carries no
information at all. A column with as many distinct values as there are rows is an
identifier rather than a measurement, which would be a different kind of problem and a
worse one.

One file gets read in full in both passes. It is a single file, so it costs little, and
reading a truncated version of it would give different null counts and different ranges,
which would then show up at the end as a mismatch that means nothing.

In [ ]:
def inspect_files_and_reference(train_dir, test_dir):
    section("Files, and one of them read in full")

    train_files = sorted(Path(train_dir).glob("*.csv"))
    test_files = sorted(Path(test_dir).glob("*.csv"))
    all_files = train_files + test_files

    print(f"train files : {len(train_files)}")
    print(f"test files  : {len(test_files)}")
    print(f"total       : {len(all_files)}   (expected {EXPECTED_FILES})")
    if len(all_files) != EXPECTED_FILES:
        print(f"MISMATCH: found {len(all_files)}, not {EXPECTED_FILES}. Check the paths above.")
    if not train_files:
        raise FileNotFoundError(f"no CSV files under {train_dir}")
    print()

    reference_file = train_files[0]
    ref = pd.read_csv(reference_file)

    print(f"reference file : {reference_file.name}")
    print(f"shape          : {ref.shape[0]:,} rows x {ref.shape[1]} columns")
    print()

    description = inv.describe_columns(ref)
    print(description.to_string(index=False))

    single_valued = description.loc[description["n_unique"] <= 1, "column"].tolist()
    row_unique = description.loc[description["n_unique"] == len(ref), "column"].tolist()
    with_nulls = description.loc[description["nulls"] > 0, "column"].tolist()

    print()
    print(f"single-valued in this file : {single_valued or 'none'}")
    print(f"distinct once per row      : {row_unique or 'none'}")
    print(f"holding any nulls          : {with_nulls or 'none'}")

    result = {
        "train_files": train_files,
        "test_files": test_files,
        "all_files": all_files,
        "n_train_files": len(train_files),
        "n_test_files": len(test_files),
        "n_files": len(all_files),
        "reference_file": reference_file,
        "reference_rows": int(len(ref)),
        "reference_columns": list(ref.columns),
        "reference_dtypes": {c: str(t) for c, t in ref.dtypes.items()},
        "column_description": description,
        "single_valued_in_reference": single_valued,
        "unique_per_row_in_reference": row_unique,
        "null_bearing_in_reference": with_nulls,
    }
    del ref
    return result

Next the two questions that decide what can be built at all. Is there a clock in the data,
and is there anything identifying an endpoint.

Both are answered by searching the column names, and both searches run two ways. A
whole-word match is a real hit. A letters-anywhere match is not, because `ts` sits inside
plenty of ordinary words and `ip` sits inside more, so those are printed separately rather
than counted as findings.

The answers are structural. They are not something a better model or a cleverer feature
can work around, so the step ends by spelling out what each one forecloses rather than
leaving me to remember it later.

In [ ]:
def answer_structural_questions(reference_columns, cfg):
    section("Is there a clock, and is there anything naming an endpoint")

    time_hits = inv.search_columns(reference_columns, inv.TIME_KEYWORDS)
    endpoint_hits = inv.search_columns(reference_columns, inv.ENDPOINT_KEYWORDS)
    has_timestamp = bool(time_hits["word_hits"])
    has_endpoints = bool(endpoint_hits["word_hits"])

    for name, keywords, hits in (
        ("timestamp", inv.TIME_KEYWORDS, time_hits),
        ("endpoint identifier", inv.ENDPOINT_KEYWORDS, endpoint_hits),
    ):
        print(f"searching {len(reference_columns)} column names for a {name}")
        print(f"  keywords : {', '.join(keywords)}")
        for col, kws in hits["word_hits"].items():
            print(f"  whole word   : {col}   [{', '.join(kws)}]")
        for col, kws in hits["substring_hits"].items():
            print(f"  letters only : {col}   [{', '.join(kws)}]   coincidence, not a field")
        if not hits["word_hits"] and not hits["substring_hits"]:
            print("  nothing matched, on either reading")
        print()

    print("ANSWER   timestamp column present      :", "YES" if has_timestamp else "NO")
    print("ANSWER   endpoint identifiers present  :", "YES" if has_endpoints else "NO")
    print()
    print("What that means for what can be built")
    print()

    if has_timestamp:
        print("A clock exists. Records can be sorted by time, the gap between two records is")
        print("measurable, and a window can be cut on elapsed time rather than on row")
        print("position. Sort order then has to be set explicitly and never assumed.")
    else:
        print("There is no clock. The only ordering available is the order the rows appear")
        print("in the file, which is the order the feature extractor wrote them. That order")
        print("means something inside a file and nothing across files, so a window must")
        print("never span two files, and the row index must never be shuffled before")
        print("windowing.")

    print()

    if has_endpoints:
        print("Endpoint identifiers exist. Records can be grouped per host or per flow")
        print("before windowing, and there are nodes and edges available for a graph. They")
        print("are also the most obvious shortcut in the data, so any model reading them has")
        print("to be checked for learning the address instead of the behaviour.")
    else:
        print("Nothing identifies a sender or a receiver. Records cannot be grouped into")
        print("flows or per-host streams, and there are no nodes from which to build a")
        print("communication graph, so graph modelling is off the table for this dataset.")
        print("A sequence here is a run of consecutive rows from one file, nothing finer.")

    if not has_timestamp and not has_endpoints:
        window, stride = cfg["sequence"]["window"], cfg["sequence"]["stride"]
        print()
        print("Both answers are negative, which fixes the definition used from here on:")
        print(f"a sequence is {window} consecutive rows from a single file, taken every")
        print(f"{stride} rows, and never crossing a file boundary.")

    return {
        "has_timestamp": has_timestamp,
        "has_endpoint_identifiers": has_endpoints,
        "timestamp_search": {"keywords": inv.TIME_KEYWORDS, **time_hits},
        "endpoint_search": {"keywords": inv.ENDPOINT_KEYWORDS, **endpoint_hits},
    }

One file agreeing with itself proves nothing about the other seventy-one. Reading only the
header costs almost nothing, so this step reads the header of every file and compares its
column list against the reference exactly, order included. Any file that differs gets
printed with what it is missing and what it has extra.

Headers are cheap, so this runs in both passes.

In [ ]:
def compare_headers(all_files, reference_columns, reference_file):
    section("Do all the files carry the same columns")

    differences = []
    for path in all_files:
        cols = inv.header_of(path)
        if cols == reference_columns:
            continue
        missing = [c for c in reference_columns if c not in cols]
        extra = [c for c in cols if c not in reference_columns]
        differences.append(
            {
                "file": path.name,
                "n_columns": len(cols),
                "missing": missing,
                "extra": extra,
                "same_columns_reordered": not missing and not extra,
            }
        )

    identical = not differences

    print(f"reference : {reference_file.name}, {len(reference_columns)} columns")
    print(f"compared  : {len(all_files)} files")
    print()
    if identical:
        print(f"ANSWER   all {len(all_files)} files carry the same {len(reference_columns)}")
        print("         columns in the same order. One column list holds for the whole")
        print("         dataset.")
    else:
        print(f"ANSWER   {len(differences)} file(s) differ from the reference:")
        for d in differences:
            print(f"  {d['file']}   ({d['n_columns']} columns)")
            if d["missing"]:
                print(f"      missing : {d['missing']}")
            if d["extra"]:
                print(f"      extra   : {d['extra']}")
            if d["same_columns_reordered"]:
                print("      same columns, different order")

    return {
        "schema_identical_across_files": identical,
        "files_differing_from_reference": differences,
    }

Now how the recordings are organised, which is readable from the file names. A name like
`TCP_IP-DDoS-ICMP1_train.pcap.csv` says three things at once: the class is DDoS-ICMP, the
recording is number one of that class's sessions, and this file is its training part.
Stripping the trailing digits gives the class, stripping the suffix gives the recording,
and `ARP_Spoofing` is the file name spelling of the class the paper calls Spoofing.

That rule lives in `src/captures.py` as `parse_capture` and is imported rather than
retyped, so every notebook after this one groups the files the same way. If the rule turns
out to be wrong, I want it wrong in one place.

The table is the whole map: for each class, which recordings appear in the training
partition, which appear in the test partition, and how many of each. I check the class
count against nineteen because that is what the dataset is documented to contain, and a
disagreement means my parsing rule is wrong rather than the paper. This reads no data at
all, only names, which is why it belongs in the fast pass. It is also the check most
likely to catch a real mistake.

In [ ]:
def map_captures(all_files):
    section("Classes and recordings, read from the file names")

    parsed = pd.DataFrame([{"file": p.name, **cap.parse_capture(p.name)} for p in all_files])
    parsed["path"] = [str(p) for p in all_files]

    print("how the rule reads a few of the names:")
    print(parsed[["file", "capture_id", "label", "partition"]].head(6).to_string(index=False))
    print()

    def capture_list(group, partition):
        return sorted(group.loc[group["partition"] == partition, "capture_id"].unique())

    rows = []
    for label, group in parsed.groupby("label"):
        train_caps = capture_list(group, "train")
        test_caps = capture_list(group, "test")
        rows.append(
            {
                "label": label,
                "n_train_captures": len(train_caps),
                "n_test_captures": len(test_caps),
                "captures_train": train_caps,
                "captures_test": test_caps,
            }
        )

    classes = pd.DataFrame(rows).sort_values("label").reset_index(drop=True)

    printable = classes.assign(
        captures_train=classes["captures_train"].map(", ".join),
        captures_test=classes["captures_test"].map(", ".join),
    )
    print(printable.to_string(index=False))
    print()

    n_classes = len(classes)
    n_captures = int(parsed["capture_id"].nunique())

    print(f"distinct classes    : {n_classes}   (expected 19)")
    print(f"distinct recordings : {n_captures}")
    print(f"train files         : {int((parsed['partition'] == 'train').sum())}")
    print(f"test files          : {int((parsed['partition'] == 'test').sum())}")

    assert n_classes == 19, f"expected 19 classes, found {n_classes}: {sorted(classes['label'])}"
    print()
    print("19 classes confirmed.")

    return {"parsed": parsed, "classes": classes, "n_classes": n_classes, "n_captures": n_captures}

The number of training recordings a class has decides what kind of split is even possible
for it, so the tier comes straight off that table.

If a class was recorded more than once in the training partition, a whole recording can be
held out and the model tested on a session it has never seen. Call that tier A. If it was
recorded once, there is nothing to hold out, and the only split available is a cut inside
that single recording, where the training rows and the test rows come from the same
session minutes apart. Call that tier B.

Those are two different experiments and their scores are not comparable, which is why the
classes are labelled rather than pooled. I check the counts against eight and eleven for
the same reason as the nineteen above: if the split of classes across the two tiers comes
out differently, my reading of the file names is wrong somewhere.

In [ ]:
def assign_tiers(classes):
    section("Tiers, and what kind of split each class allows")

    classes = classes.copy()
    classes["tier"] = np.where(classes["n_train_captures"] > 1, "A", "B")

    tier_a = classes.loc[classes["tier"] == "A", "label"].tolist()
    tier_b = classes.loc[classes["tier"] == "B", "label"].tolist()

    print("Tier A, more than one training recording, whole recordings can be held out")
    for label in tier_a:
        n = int(classes.loc[classes["label"] == label, "n_train_captures"].iloc[0])
        print(f"  {label:<28} {n} training recordings")
    print()
    print("Tier B, one training recording, only a split inside that recording is possible")
    for label in tier_b:
        print(f"  {label}")
    print()
    print(f"tier A : {len(tier_a):>2}   (expected 8)")
    print(f"tier B : {len(tier_b):>2}   (expected 11)")

    if (len(tier_a), len(tier_b)) != (8, 11):
        print()
        print(f"DIFFERS FROM EXPECTATION: {len(tier_a)} in A and {len(tier_b)} in B.")
        print("Tier A:", tier_a)
        print("Tier B:", tier_b)

    assert (len(tier_a), len(tier_b)) == (8, 11), (
        f"expected 8 in tier A and 11 in tier B, got {len(tier_a)} and {len(tier_b)}"
    )
    print()
    print("Tier split confirmed at 8 and 11.")

    return {"classes": classes, "tier_a": tier_a, "tier_b": tier_b}

Sizes next, and this is the first step the fast pass does not take.

Counting the rows means touching every byte of every file. Parsing seventy-two files just
to count their lines would be worse, so I count newlines instead and check that shortcut
against pandas on one file before trusting it on the rest, but it is still a pass over the
whole corpus. There is no cheap version of it that is also correct: counting the rows of a
sample tells me the size of the sample.

So in the fast pass it is skipped and says so, and everything that depends on it, which is
the class sizes, the three imbalance ratios and both figures, is skipped with it.

The gap between the largest and the smallest class is the number that decides how results
get reported, so the full pass takes it three times, once for each grouping the benchmark
is scored at. Collapsing nineteen classes into six and then into two moves that gap a long
way, and a score that looks fine at two classes can hide a class that is never predicted
at all.

In [ ]:
def count_rows(all_files, parsed, classes, reference_file, *, fast):
    if fast:
        skipped("Rows per file and per class")
        print("Counting rows reads every byte of all 72 files. A count taken from a sample")
        print("is the size of the sample and nothing else, so there is no cheap version.")
        print("Class sizes, the three imbalance ratios and both figures go with it.")
        return {
            "rows_counted": False,
            "row_counts": None,
            "total_rows": None,
            "class_sizes": None,
            "rows_per_group6": None,
            "rows_per_group2": None,
            "imbalance": None,
        }

    section("Rows per file and per class")

    probe_fast, probe_pandas = inv.verify_row_count(reference_file)
    print(f"newline count vs pandas on {reference_file.name} : {probe_fast:,} vs {probe_pandas:,}")
    if probe_fast == probe_pandas:
        print("They agree, so newline counting is safe for the remaining files.")
    else:
        print("They disagree, so a field contains a newline and the counts below are wrong.")
    print()

    row_counts = {p.name: inv.count_data_rows(p) for p in all_files}
    parsed = parsed.copy()
    parsed["rows"] = [row_counts[p.name] for p in all_files]
    total_rows = int(parsed["rows"].sum())

    print("rows per file")
    print(
        parsed[["file", "label", "capture_id", "partition", "rows"]]
        .sort_values(["label", "partition", "capture_id"])
        .to_string(index=False)
    )
    print()

    class_sizes = (
        parsed.groupby("label")["rows"].sum().rename("rows").reset_index().merge(classes, on="label")
    )
    class_sizes["pct_of_total"] = (100 * class_sizes["rows"] / total_rows).round(3)
    class_sizes["group6"] = class_sizes["label"].map(cap.group_six)
    class_sizes["group2"] = class_sizes["label"].map(cap.group_two)
    class_sizes = class_sizes.sort_values("rows", ascending=False).reset_index(drop=True)

    unmapped = class_sizes.loc[class_sizes["group6"] == "UNMAPPED", "label"].tolist()
    assert not unmapped, f"labels the 6-class map does not cover: {unmapped}"

    print("rows per class, largest first")
    print(
        class_sizes[
            ["label", "tier", "rows", "pct_of_total", "n_train_captures", "n_test_captures"]
        ].to_string(index=False)
    )
    print()
    print(f"total rows across {len(all_files)} files : {total_rows:,}")
    print()

    def grouped(column):
        frame = (
            class_sizes.groupby(column)
            .agg(classes=("label", "count"), rows=("rows", "sum"))
            .sort_values("rows", ascending=False)
        )
        frame["pct_of_total"] = (100 * frame["rows"] / total_rows).round(3)
        return frame

    six, two = grouped("group6"), grouped("group2")
    largest, smallest = class_sizes.iloc[0], class_sizes.iloc[-1]
    imbalance = {
        "19_class": float(largest["rows"]) / max(float(smallest["rows"]), 1.0),
        "6_class": float(six["rows"].max()) / max(float(six["rows"].min()), 1.0),
        "2_class": float(two["rows"].max()) / max(float(two["rows"].min()), 1.0),
    }

    print("6-class grouping")
    print(six.to_string())
    print()
    print("2-class grouping")
    print(two.to_string())
    print()
    print("largest class to smallest class")
    print(f"  19-class : {imbalance['19_class']:>10,.1f} to 1    "
          f"{largest['label']} vs {smallest['label']}")
    print(f"   6-class : {imbalance['6_class']:>10,.1f} to 1    {six.index[0]} vs {six.index[-1]}")
    print(f"   2-class : {imbalance['2_class']:>10,.1f} to 1    {two.index[0]} vs {two.index[-1]}")
    print()

    share = float(largest["rows"]) / total_rows
    majority_macro = (2 * share / (1 + share)) / 19
    print(f"A model that always answers {largest['label']} and nothing else would be right")
    print(f"{100 * share:.1f}% of the time and score a macro-F1 of {majority_macro:.3f} across")
    print("the nineteen classes. The distance between those two numbers is why macro-F1 is")
    print("the primary metric here and why accuracy is never reported on its own.")

    return {
        "rows_counted": True,
        "row_counts": row_counts,
        "parsed": parsed,
        "total_rows": total_rows,
        "class_sizes": class_sizes,
        "rows_per_group6": {k: int(v) for k, v in six["rows"].items()},
        "rows_per_group2": {k: int(v) for k, v in two["rows"].items()},
        "imbalance": {k: round(v, 3) for k, v in imbalance.items()},
        "largest_class": largest["label"],
        "smallest_class": smallest["label"],
    }

The second step the fast pass does not take, and the one I most want to come back
negative. It is looking for two different things.

The first is a column that never changes anywhere in the dataset. A column like that
separates nothing from nothing, so it can be dropped without losing anything. Worth
knowing about, but harmless.

The second is the one that matters. If a column holds a single value for the whole of one
recording and a different single value in the next recording, then it is not describing
the traffic at all, it is naming the session. Each attack class here was recorded in its
own session, so a column that names the session also names the class. A model reading it
would score well on this dataset while having learned nothing about attacks, and the score
would collapse the moment it met traffic recorded anywhere else. That is the failure I
want to rule out before I trust any number this project produces.

This is the step that most obviously cannot be sampled. A column that holds still through
the first one percent of a file and moves through the rest would be flagged as constant by
a sample, and the flag would be wrong in the most damaging direction: it would announce a
problem that is not there, or, run the other way, it would let one through. So the fast
pass does not attempt it and says so.

I take a single file as the unit of a recording, because a file is one continuous stretch
of capture whereas a group of files is only a recording if I have grouped it correctly.
The step ends by repeating the count with files sharing a recording name treated as one,
to check the answer does not depend on that choice.

In [ ]:
def scan_constancy(all_files, parsed, row_counts, total_rows, *, fast):
    if fast:
        skipped("Columns that never change")
        print("This one reads every row of every file by definition. A column can hold one")
        print("value through the head of a file and move through the rest, so a sampled")
        print("answer would be wrong in whichever direction happened to be convenient.")
        return {
            "constancy_scanned": False,
            "rows_scanned": None,
            "constancy_report": None,
            "constant_everywhere": None,
            "recording_identifying": None,
            "constant_in_some_recordings_only": None,
            "grouping_agrees": None,
        }

    section("Columns that never change")

    started = time.time()
    print(f"reading every row of {len(all_files)} files")
    scans = cap.scan_captures(all_files, frac=1.0, known_rows=row_counts)
    rows_scanned = int(sum(s["n_rows_read"] for s in scans.values()))
    print(f"done in {time.time() - started:.0f}s")
    print()
    print(f"rows read : {rows_scanned:,} of {total_rows:,}")
    if rows_scanned != total_rows:
        print("The parser and the newline count disagree on the total. Something in the")
        print("data contains a newline, and what follows is not trustworthy.")
    print()

    # Two planted controls, one of each kind, to show the check can find them.
    DEAD, MARKER = "__planted_dead_column__", "__planted_recording_marker__"
    planted = {
        name: {
            **scan,
            "columns": {
                **scan["columns"],
                DEAD: {"n_unique": 1, "value": 0.0},
                MARKER: {"n_unique": 1, "value": i},
            },
        }
        for i, (name, scan) in enumerate(scans.items())
    }
    control = cap.constancy_report(planted).set_index("column")
    dead_ok = bool(control.loc[DEAD, "constant_everywhere"])
    marker_ok = bool(control.loc[MARKER, "recording_identifying"])
    print(f"control, one value everywhere    : classified constant everywhere = {dead_ok}")
    print(f"control, one value per recording : classified recording-naming    = {marker_ok}")
    assert dead_ok and marker_ok, "the constancy check failed on its own planted controls"
    print("Both controls land where they should, so the check is working.")
    print()

    SHOW = [
        "column",
        "constant_in_recordings",
        "of_recordings",
        "distinct_values_across_recordings",
    ]
    report = cap.constancy_report(scans)

    dead_columns = report.loc[report["constant_everywhere"], "column"].tolist()
    identifying = report.loc[report["recording_identifying"], "column"].tolist()
    near_miss = report[
        (report["constant_in_recordings"] > 0)
        & ~report["constant_everywhere"]
        & ~report["recording_identifying"]
    ]

    print("=" * 79)
    print("FINDING 1   columns that never change anywhere in the dataset")
    print("=" * 79)
    if dead_columns:
        print(report[report["constant_everywhere"]][SHOW + ["values"]].to_string(index=False))
        print()
        print(f"{len(dead_columns)} column(s) hold one value across all {len(scans)} files.")
        print("They separate nothing and can be dropped.")
    else:
        print("None. Every column takes more than one value somewhere.")
    print()

    print("=" * 79)
    print("FINDING 2   columns constant inside a recording but different between recordings")
    print("=" * 79)
    if identifying:
        print(report[report["recording_identifying"]][SHOW + ["values"]].to_string(index=False))
        print()
        print(f"{len(identifying)} column(s) name the recording rather than measure traffic.")
        print("They must be dropped before any model is fitted, and any earlier score that")
        print("used them means nothing.")
    else:
        print("None. No column sits still for a whole recording and then moves to a new")
        print("value in the next one, so no single column acts as a name for the session a")
        print("row came from. That does not clear the dataset. It rules out the crudest")
        print("version of the problem, where one column is the giveaway. A shift in the")
        print("distribution of a column that varies within every recording would carry the")
        print("same information more quietly, and nothing here would catch it.")
    print()

    print("near misses, constant in some recordings but not all")
    print(near_miss[SHOW].to_string(index=False) if len(near_miss) else "  none")
    print()

    print("every column, ordered by how often it held still")
    print(report[SHOW + ["constant_everywhere", "recording_identifying"]].to_string(index=False))
    print()

    # Does the answer survive grouping the files by recording name?
    capture_of = dict(zip(parsed["file"], parsed["capture_id"]))
    by_capture = cap.constancy_report(scans, capture_of)
    grouped_dead = by_capture.loc[by_capture["constant_everywhere"], "column"].tolist()
    grouped_ident = by_capture.loc[by_capture["recording_identifying"], "column"].tolist()

    print(f"unit = one file           : {len(scans)} recordings, "
          f"{len(dead_columns)} dead, {len(identifying)} recording-naming")
    print(f"unit = one recording name : {parsed['capture_id'].nunique()} recordings, "
          f"{len(grouped_dead)} dead, {len(grouped_ident)} recording-naming")
    print()

    agrees = set(grouped_dead) == set(dead_columns) and set(grouped_ident) == set(identifying)
    if agrees:
        print("The two readings agree, so the finding does not depend on how the files were")
        print("grouped into recordings.")
    else:
        print("The two readings disagree, so the grouping is telling me something about the")
        print("files that their names do not:")
        print(f"  dead only when grouped     : {sorted(set(grouped_dead) - set(dead_columns))}")
        print(f"  dead only when ungrouped   : {sorted(set(dead_columns) - set(grouped_dead))}")
        print(f"  naming only when grouped   : {sorted(set(grouped_ident) - set(identifying))}")
        print(f"  naming only when ungrouped : {sorted(set(identifying) - set(grouped_ident))}")

    return {
        "constancy_scanned": True,
        "rows_scanned": rows_scanned,
        "constancy_report": report,
        "constant_everywhere": dead_columns,
        "recording_identifying": identifying,
        "constant_in_some_recordings_only": near_miss["column"].tolist(),
        "grouping_agrees": bool(agrees),
    }

The last step of the pass groups the columns into timing, protocol, statistical and other.
Later notebooks need to be able to ask what a model scores using only the timing columns,
or only the protocol ones, and that question needs the groups fixed in a file rather than
retyped each time.

This grouping is guesswork from spelling and nothing else. No column is opened to see what
it actually measures, so the file records itself as a draft and lists every column that
matched more than one rule. Rate is the obvious problem: it reads as timing to me because
it is a per-second quantity, but it is computed from a count, and a name like `ack_count`
has an equal claim on protocol and on statistical. I would rather flag those than quietly
decide them.

It works from the column names alone, so it runs in both passes and is one more thing the
comparison at the end can check.

In [ ]:
def build_feature_families(reference_columns, reference_file, git_sha, run_date):
    section("A first grouping of the columns, from their names")

    assignment = inv.assign_families(reference_columns)

    for name, cols in assignment["families"].items():
        print(f"{name:<12} {len(cols):>3}")
        for col in cols:
            print(f"             {col}")
    print()
    print(f"label columns : {assignment['labels'] or 'none found by name'}")
    print(f"ambiguous     : {len(assignment['ambiguous'])} column(s) matched more than one rule")
    for col, names in assignment["ambiguous"].items():
        print(f"                {col}  ->  {' / '.join(names)}   (assigned to {names[0]})")

    text = inv.render_feature_families_yaml(
        assignment,
        source_file=reference_file.name,
        git_sha=git_sha,
        run_date=run_date,
        generated_by="AG_PRAXIS_NB01_dataset_inventory.ipynb",
    )
    return {"family_assignment": assignment, "feature_families_yaml": text}

Two pictures, because the two numbers I will keep coming back to are hard to hold in a
table of nineteen rows.

The first is class size. It has to be on a log scale, because that is the only scale on
which classes spanning three orders of magnitude fit on one axis at all, and the fact that
a linear axis is unusable is itself the point. The second is how many recordings each
class has, which is where the tier comes from.

Both are coloured by tier, so the two figures can be read against each other: the classes
with the fewest rows are also the ones with the fewest recordings, and those are the same
classes that will be hardest to split honestly.

Both need the row counts, so they are drawn by the full pass only. Drawing a figure from
the fast pass would produce a picture that looks like a result and is not one.

In [ ]:
import matplotlib.pyplot as plt

INK, MUTED, GRID, RULE = "#0b0b0b", "#52514e", "#e6e5e1", "#c9c8c3"
TIER_COLOUR = {"A": "#2a78d6", "B": "#eb6834"}
TIER_LABEL = {
    "A": "Tier A, more than one training recording",
    "B": "Tier B, one training recording",
}


def _style(ax):
    ax.grid(axis="x", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(RULE)
    ax.tick_params(colors=MUTED, length=0)
    for label in ax.get_yticklabels():
        label.set_color(INK)


def _tier_legend(ax):
    handles = [
        plt.Rectangle((0, 0), 1, 1, color=TIER_COLOUR[t], label=TIER_LABEL[t]) for t in ("A", "B")
    ]
    legend = ax.legend(handles=handles, loc="lower right", frameon=False, fontsize=9)
    for text in legend.get_texts():
        text.set_color(MUTED)


def make_figures(class_sizes, total_rows, ratio_19, out_dir):
    section("Figures")

    order = class_sizes.sort_values("rows")
    colours = [TIER_COLOUR[t] for t in order["tier"]]
    written = []

    fig, ax = plt.subplots(figsize=(9, 7))
    bars = ax.barh(order["label"], order["rows"], color=colours, height=0.68)
    ax.set_xscale("log")
    ax.set_xlim(right=order["rows"].max() * 4)
    ax.bar_label(
        bars, labels=[f"{int(v):,}" for v in order["rows"]], padding=4, color=MUTED, fontsize=8
    )
    ax.set_xlabel("rows, log scale", color=MUTED, fontsize=9)
    ax.set_title(
        f"Rows per class, {total_rows:,} rows over 19 classes, "
        f"{ratio_19:,.0f} to 1 largest to smallest",
        color=INK,
        fontsize=11,
        loc="left",
        pad=12,
    )
    _style(ax)
    _tier_legend(ax)
    path = out_dir / "NB01_class_sizes.png"
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    written.append(path)
    print(f"wrote {path}")
    plt.show()

    fig, ax = plt.subplots(figsize=(9, 7))
    bars = ax.barh(order["label"], order["n_train_captures"], color=colours, height=0.68)
    ax.bar_label(
        bars,
        labels=[
            f"{int(t)} train, {int(s)} test"
            for t, s in zip(order["n_train_captures"], order["n_test_captures"])
        ],
        padding=4,
        color=MUTED,
        fontsize=8,
    )
    ax.set_xlim(right=order["n_train_captures"].max() + 2.6)
    ax.set_xlabel("distinct training recordings", color=MUTED, fontsize=9)
    ax.set_title(
        "Recordings per class, the count the tier is read from",
        color=INK,
        fontsize=11,
        loc="left",
        pad=12,
    )
    _style(ax)
    _tier_legend(ax)
    path = out_dir / "NB01_recordings_per_class.png"
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    written.append(path)
    print(f"wrote {path}")
    plt.show()

    return written

Everything a pass computes then goes into one file. The point of writing it down is that
no later notebook should have to reread seventy-two files to find out how many classes
there are, and none of them should be allowed to assume it either.

The document says which mode wrote it, and the steps a fast pass skipped are recorded as
`null` next to a flag saying they were not run. That way a missing number is never
mistaken for a measured zero, and the fast pass's file can never be read as a result.

In [ ]:
def inventory_document(r, *, git_sha, git_dirty, run_date, seed):
    class_sizes = r["class_sizes"]
    doc = {
        "generated_by": "AG_PRAXIS_NB01_dataset_inventory.ipynb",
        "generated_on": run_date,
        "git_sha": git_sha,
        "git_dirty": git_dirty,
        "seed": seed,
        "mode": r["mode"],
        "is_fast_pass": r["fast"],
        "enterable_in_ledger": not r["fast"],
        "rows_counted": r["rows_counted"],
        "constancy_scanned": r["constancy_scanned"],
        "rows_scanned": r["rows_scanned"],
        "reference_file": r["reference_file"].name,
        "n_files": r["n_files"],
        "n_files_expected": EXPECTED_FILES,
        "n_train_files": r["n_train_files"],
        "n_test_files": r["n_test_files"],
        "n_columns": len(r["reference_columns"]),
        "n_classes": r["n_classes"],
        "n_captures": r["n_captures"],
        "total_rows": r["total_rows"],
        "columns": r["reference_columns"],
        "dtypes": r["reference_dtypes"],
        "column_description": r["column_description"].to_dict(orient="records"),
        "has_timestamp": r["has_timestamp"],
        "has_endpoint_identifiers": r["has_endpoint_identifiers"],
        "timestamp_search": r["timestamp_search"],
        "endpoint_search": r["endpoint_search"],
        "schema_identical_across_files": r["schema_identical_across_files"],
        "files_differing_from_reference": r["files_differing_from_reference"],
        "tiers": {"A": r["tier_a"], "B": r["tier_b"]},
        "rows_per_file": r["row_counts"],
        "rows_per_group6": r["rows_per_group6"],
        "rows_per_group2": r["rows_per_group2"],
        "imbalance_ratio": r["imbalance"],
        "figures": [p.name for p in r["figures"]],
    }

    source = class_sizes if class_sizes is not None else r["classes"]
    doc["classes"] = {
        row["label"]: {
            "tier": row["tier"],
            "group6": row.get("group6", cap.group_six(row["label"])),
            "group2": row.get("group2", cap.group_two(row["label"])),
            "rows": int(row["rows"]) if class_sizes is not None else None,
            "pct_of_total": float(row["pct_of_total"]) if class_sizes is not None else None,
            "n_train_captures": int(row["n_train_captures"]),
            "n_test_captures": int(row["n_test_captures"]),
            "captures_train": row["captures_train"],
            "captures_test": row["captures_test"],
        }
        for _, row in source.iterrows()
    }

    report = r["constancy_report"]
    doc["constant_columns"] = {
        "scanned": r["constancy_scanned"],
        "unit": "one file",
        "constant_everywhere": r["constant_everywhere"],
        "recording_identifying": r["recording_identifying"],
        "constant_in_some_recordings_only": r["constant_in_some_recordings_only"],
        "grouping_by_recording_name_agrees": r["grouping_agrees"],
        "per_column": None
        if report is None
        else [
            {
                "column": row["column"],
                "constant_in_recordings": int(row["constant_in_recordings"]),
                "of_recordings": int(row["of_recordings"]),
                "distinct_values_across_recordings": int(row["distinct_values_across_recordings"]),
                "constant_everywhere": bool(row["constant_everywhere"]),
                "recording_identifying": bool(row["recording_identifying"]),
                "values": row["values"][:20],
            }
            for _, row in report.iterrows()
        ],
    }
    return doc

`run_inventory` calls the steps in order and hands each one what the previous ones
produced. It takes a single argument, and the only thing that argument changes is whether
the two expensive steps run or announce themselves skipped. Nothing is computed a
different way in fast mode, which is what makes the comparison at the end meaningful: any
value both passes produced was produced by exactly the same code.

It writes its files before returning, into the folder belonging to its own mode, and only
the full pass touches the copy of `feature_families.yaml` inside the repository.

In [ ]:
def run_inventory(fast: bool) -> dict:
    mode = "fast" if fast else "full"
    out_dir = OUT_DIRS[mode]
    out_dir.mkdir(parents=True, exist_ok=True)

    started = time.time()
    r = {"mode": mode, "fast": fast, "out_dir": out_dir}

    r.update(inspect_files_and_reference(TRAIN_DIR, TEST_DIR))
    r.update(answer_structural_questions(r["reference_columns"], CFG))
    r.update(compare_headers(r["all_files"], r["reference_columns"], r["reference_file"]))
    r.update(map_captures(r["all_files"]))
    r.update(assign_tiers(r["classes"]))
    r.update(
        count_rows(r["all_files"], r["parsed"], r["classes"], r["reference_file"], fast=fast)
    )
    r.update(
        scan_constancy(
            r["all_files"], r["parsed"], r["row_counts"], r["total_rows"], fast=fast
        )
    )
    r.update(
        build_feature_families(
            r["reference_columns"], r["reference_file"], GIT_SHA, RUN_DATE
        )
    )

    r["figures"] = (
        []
        if fast
        else make_figures(
            r["class_sizes"], r["total_rows"], r["imbalance"]["19_class"], out_dir
        )
    )

    section(f"Writing the {mode} pass to {out_dir}")

    doc = inventory_document(r, git_sha=GIT_SHA, git_dirty=GIT_DIRTY, run_date=RUN_DATE, seed=SEED)
    inventory_path = out_dir / "dataset_inventory.json"
    inventory_text = json.dumps(cap.jsonable(doc), indent=2, default=str) + "\n"
    inventory_path.write_text(inventory_text)
    print(f"wrote {inventory_path}   ({len(inventory_text):,} bytes)")

    families_path = out_dir / "feature_families.yaml"
    families_path.write_text(r["feature_families_yaml"])
    print(f"wrote {families_path} as a draft")

    if not fast:
        repo_copy = REPO_ROOT / "config" / "feature_families.yaml"
        repo_copy.write_text(r["feature_families_yaml"])
        print(f"wrote {repo_copy}")
        if IN_COLAB:
            print("The repository copy goes when the session does. Move the Drive copy into")
            print("config/feature_families.yaml from the Mac and commit it there.")
    else:
        print("The fast pass does not touch config/feature_families.yaml in the repository.")

    r["document"] = doc
    r["inventory_path"] = inventory_path
    r["families_path"] = families_path
    r["elapsed_s"] = time.time() - started
    return r

Both passes run here, fast first. If the fast one raises, the full one never starts, which
is the whole point: a wrong path or a broken parsing rule costs a minute rather than the
best part of an hour.

In [ ]:
def banner(lines):
    print()
    print("#" * 79)
    for line in lines:
        print(f"#  {line:<75}#")
    print("#" * 79)


BANNERS = {
    "fast": [
        "FAST PASS",
        "cheap checks only, a test of the code and not a measurement of the data",
        "never entered in the ledger",
    ],
    "full": [
        "FULL PASS",
        "every step, every row",
        "this is the pass the repository takes",
    ],
}

results = {}
for fast in (True, False):
    name = "fast" if fast else "full"
    banner(BANNERS[name])
    results[name] = run_inventory(fast)

banner([
    f"fast pass {results['fast']['elapsed_s']:.0f}s, "
    f"full pass {results['full']['elapsed_s']:.0f}s"
])

Now the two passes side by side, on every value both of them produced.

None of these depend on how much of a file was read. They come from file names, from
headers, and from one file that both passes read in full, so the two answers have to be
identical. If any row below says MISMATCH it means the fast path computed something a
different way, which is a bug in this notebook. It is not a finding about the dataset and
it must not be reported as one.

The steps the fast pass skipped are listed separately underneath, so it is clear what is
being compared and what is not.

In [ ]:
FAST, FULL = results["fast"], results["full"]


def frame(value):
    return None if value is None else value.to_dict(orient="records")


COMPARISONS = [
    ("files found", lambda r: r["n_files"]),
    ("train files", lambda r: r["n_train_files"]),
    ("test files", lambda r: r["n_test_files"]),
    ("reference file", lambda r: r["reference_file"].name),
    ("rows in reference file", lambda r: r["reference_rows"]),
    ("column count", lambda r: len(r["reference_columns"])),
    ("column list and order", lambda r: r["reference_columns"]),
    ("column dtypes", lambda r: r["reference_dtypes"]),
    ("column description table", lambda r: frame(r["column_description"])),
    ("timestamp column present", lambda r: r["has_timestamp"]),
    ("timestamp name search", lambda r: r["timestamp_search"]),
    ("endpoint identifiers present", lambda r: r["has_endpoint_identifiers"]),
    ("endpoint name search", lambda r: r["endpoint_search"]),
    ("same columns in every file", lambda r: r["schema_identical_across_files"]),
    ("files differing from reference", lambda r: r["files_differing_from_reference"]),
    ("class count", lambda r: r["n_classes"]),
    ("recording count", lambda r: r["n_captures"]),
    ("class to recording map", lambda r: frame(r["classes"].drop(columns="tier"))),
    ("tier assignment", lambda r: frame(r["classes"][["label", "tier"]])),
    ("tier A membership", lambda r: r["tier_a"]),
    ("tier B membership", lambda r: r["tier_b"]),
    ("feature family assignment", lambda r: r["family_assignment"]),
    ("feature_families.yaml body", lambda r: r["feature_families_yaml"].split("git_sha")[0]),
]

mismatches = []
print(f"{'value':<34} {'result':<10} fast vs full")
print("-" * 79)
for label, get in COMPARISONS:
    a, b = get(FAST), get(FULL)
    ok = a == b
    print(f"{label:<34} {'PASS' if ok else 'MISMATCH':<10} " + ("" if ok else "see below"))
    if not ok:
        mismatches.append((label, a, b))

print("-" * 79)
print(f"{len(COMPARISONS) - len(mismatches)} of {len(COMPARISONS)} values agree")
COMPARISON_OK = not mismatches

if mismatches:
    print()
    print("The fast path and the full path disagree. That is a bug in this notebook, not")
    print("something learned about the dataset. Nothing from this run should be reported.")
    for label, a, b in mismatches:
        print()
        print(f"  {label}")
        print(f"    fast : {str(a)[:400]}")
        print(f"    full : {str(b)[:400]}")

print()
print("not compared, because the fast pass does not compute them:")
for label, ok in (
    ("row counts per file and per class", FULL["rows_counted"]),
    ("class sizes and the three imbalance ratios", FULL["rows_counted"]),
    ("the constant-column scan", FULL["constancy_scanned"]),
    ("both figures", bool(FULL["figures"])),
):
    print(f"  {label:<48} full pass ran it: {ok}")

The full pass's two files get printed here in full. Colab cannot push to the repository
from a cell, so until they are moved across by hand the only durable record of what this
run produced is the saved copy of this notebook. Printing them means the executed notebook
carries the whole result even if the artefacts folder is later cleared.

Only the full pass is printed. The fast pass's copies are on Drive under `NB01_fast` if
they are ever needed for debugging, and they are not results.

In [ ]:
for path in (FULL["inventory_path"], FULL["families_path"]):
    print("=" * 79)
    print(f"{path}   ({path.stat().st_size:,} bytes)")
    print("=" * 79)
    print(path.read_text().rstrip())
    print()

print("=" * 79)
print("figures, which cannot be printed")
print("=" * 79)
for path in FULL["figures"]:
    print(f"{path.name}   {path.stat().st_size:,} bytes")

The entry for `RESULTS_LEDGER.md`, ready to paste. It reports the full pass only. The fast
pass is a test of this notebook and has nothing to say about the data.

In [ ]:
if not COMPARISON_OK:
    status = "DO NOT ENTER, the fast and full passes disagree and the notebook has a bug"
elif GIT_DIRTY:
    status = "reference run, working tree dirty"
else:
    status = "reference run"

imbalance = FULL["imbalance"]
dead, naming = FULL["constant_everywhere"], FULL["recording_identifying"]
largest = FULL["class_sizes"].iloc[0]
smallest = FULL["class_sizes"].iloc[-1]

ledger = f'''
### NB01 — dataset inventory ({RUN_DATE})

| field | value |
|---|---|
| notebook | AG_PRAXIS_NB01_dataset_inventory.ipynb |
| run date | {RUN_DATE} |
| git sha | {GIT_SHA}{" (working tree dirty)" if GIT_DIRTY else ""} |
| seed | {SEED} |
| pass reported | full, every step over every row |
| fast/full agreement | {"all {} compared values agree".format(len(COMPARISONS)) if COMPARISON_OK else "MISMATCH, see the comparison cell"} |
| status | {status} |
| runtime | fast {FAST["elapsed_s"]:.0f}s, full {FULL["elapsed_s"]:.0f}s |
| files read | {FULL["n_files"]} of {EXPECTED_FILES} expected |
| columns | {len(FULL["reference_columns"])} |
| same columns in every file | {"yes" if FULL["schema_identical_across_files"] else "no, {} file(s) differ".format(len(FULL["files_differing_from_reference"]))} |
| timestamp column | {"yes" if FULL["has_timestamp"] else "no"} |
| endpoint identifiers | {"yes" if FULL["has_endpoint_identifiers"] else "no"} |
| classes | {FULL["n_classes"]} |
| recordings | {FULL["n_captures"]} |
| tier A / tier B | {len(FULL["tier_a"])} / {len(FULL["tier_b"])} |
| total rows | {FULL["total_rows"]:,} |
| rows scanned for constancy | {FULL["rows_scanned"]:,} |
| largest class | {largest["label"]}, {int(largest["rows"]):,} rows, {largest["pct_of_total"]}% |
| smallest class | {smallest["label"]}, {int(smallest["rows"]):,} rows, {smallest["pct_of_total"]}% |
| imbalance, 19-class | {imbalance["19_class"]:,.1f} to 1 |
| imbalance, 6-class | {imbalance["6_class"]:,.1f} to 1 |
| imbalance, 2-class | {imbalance["2_class"]:,.1f} to 1 |
| columns constant everywhere | {len(dead)}{": " + ", ".join(dead) if dead else ""} |
| columns naming the recording | {len(naming)}{": " + ", ".join(naming) if naming else ""} |
| recording grouping changes the answer | {"no" if FULL["grouping_agrees"] else "YES, see the constancy step"} |
| metrics | none, no model trained |
| artefacts | {FULL["out_dir"]}, holding dataset_inventory.json, feature_families.yaml (draft), and two figures |

Tier B classes, which admit no capture-disjoint split: {", ".join(FULL["tier_b"])}
'''

print(ledger)